# ComfyUI Colab Studio

Self-contained. Nothing to upload. **Runtime > Run all**, then use the
generate cell at the bottom or open the tunnel URL.

| Cell | What it does |
|---|---|
| 1 | Pick model + persistence options |
| 2 | Detect GPU, print what fits, choose launch flags |
| 3-6 | Install ComfyUI, link Drive, download models, write workflows |
| 7 | Start the server **in the background** and open a public URL |
| 8 | Generate an image without leaving this notebook |
| 9-10 | Logs, restart, free VRAM, disk usage |
| 11 | Handbook: model sizes, settings, error fixes |

In [ ]:
#@title 1. Config
MODE = "image"  #@param ["image", "video"]
IMAGE_MODEL = "auto"  #@param ["auto", "sdxl", "flux-dev", "flux-schnell"]
PERSIST = "outputs-only"  #@param ["outputs-only", "everything", "off"]
CONTROLNET = False  #@param {type:"boolean"}
UPSCALER = True  #@param {type:"boolean"}
PORT = 8188
COMFY_DIR = "/content/ComfyUI"
print(f"mode={MODE} model={IMAGE_MODEL} persist={PERSIST} "
      f"controlnet={CONTROLNET} upscaler={UPSCALER}")

In [ ]:
#@title 2. Preflight - GPU, disk, and what actually fits
import shutil, subprocess, torch

if not torch.cuda.is_available():
    print("!! No GPU. Runtime > Change runtime type > GPU, then rerun.")
    VRAM_GB, GPU_NAME = 0.0, "cpu"
else:
    props = torch.cuda.get_device_properties(0)
    GPU_NAME, VRAM_GB = props.name, props.total_memory / 2**30

DISK_FREE_GB = shutil.disk_usage("/content").free / 2**30
print(f"GPU:  {GPU_NAME}  ({VRAM_GB:.1f} GB VRAM)")
print(f"Disk: {DISK_FREE_GB:.0f} GB free")

In [ ]:
#@title 3. Install ComfyUI
import os
%cd /content
if not os.path.isdir(COMFY_DIR):
    !git clone https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}
%cd {COMFY_DIR}
!pip install -q -r requirements.txt
!pip install -q huggingface_hub torchsde requests
if MODE == "video":
    os.makedirs("custom_nodes", exist_ok=True)
    for url, name in [
        ("https://github.com/kijai/ComfyUI-WanVideoWrapper.git", "ComfyUI-WanVideoWrapper"),
        ("https://github.com/city96/ComfyUI-GGUF.git", "ComfyUI-GGUF"),
        ("https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git", "ComfyUI-VideoHelperSuite"),
    ]:
        d = os.path.join("custom_nodes", name)
        if not os.path.isdir(d):
            os.system(f"git clone {url} {d}")
        req = os.path.join(d, "requirements.txt")
        if os.path.isfile(req):
            os.system(f"pip install -q -r {req}")
os.makedirs("colab_studio", exist_ok=True)
open("colab_studio/__init__.py", "a").close()
print("ComfyUI installed at", COMFY_DIR)

In [ ]:
#@title 4. Persistence (Drive)
# Models stay on VM disk by default: one Flux checkpoint is 16 GB and the
# free Drive tier is 15 GB, so symlinking models/ fills the quota instantly
# and streams every read over FUSE.
import os

def _link(src, dest):
    os.makedirs(src, exist_ok=True)
    if os.path.isdir(dest) and not os.path.islink(dest):
        os.system(f'cp -rn "{dest}"/* "{src}"/ 2>/dev/null')
        os.system(f'rm -rf "{dest}"')
    if not os.path.islink(dest):
        os.symlink(src, dest)

if PERSIST != "off":
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/ComfyUI"
    subs = ["output", "user"]
    if PERSIST == "everything":
        print("!! Models on Drive: slow (FUSE) and Flux alone is 16 GB.")
        subs += ["models"]
    for sub in subs:
        _link(os.path.join(DRIVE, sub), os.path.join(COMFY_DIR, sub))
    print("Persisting:", ", ".join(subs))
else:
    print("No persistence - everything is lost when the runtime recycles.")

In [ ]:
%%writefile colab_studio/registry.py
"""Model registry. Pure data -- no network, no filesystem.

Every entry below was HEAD-verified against huggingface.co on 2026-07-25.
Sizes are real, not estimates.
"""
from __future__ import annotations

import os
from dataclasses import dataclass


@dataclass(frozen=True)
class ModelSpec:
    repo: str
    filename: str
    dest_subdir: str          # relative to ComfyUI models/
    size_gb: float
    dest_filename: str | None = None

    @property
    def target_name(self) -> str:
        """Filename as written to disk. Diffusers-layout repos all use the
        same generic filename, so those entries must rename."""
        return self.dest_filename or os.path.basename(self.filename)


UPSCALER = ModelSpec(
    repo="Kim2091/UltraSharp",
    filename="4x-UltraSharp.pth",
    dest_subdir="upscale_models",
    size_gb=0.06,
)

CONTROLNET_CANNY = ModelSpec(
    repo="diffusers/controlnet-canny-sdxl-1.0",
    filename="diffusion_pytorch_model.fp16.safetensors",
    dest_subdir="controlnet",
    size_gb=2.33,
    dest_filename="controlnet-canny-sdxl.safetensors",
)

PROFILES: dict[str, list[ModelSpec]] = {
    "sdxl": [
        ModelSpec(
            repo="stabilityai/stable-diffusion-xl-base-1.0",
            filename="sd_xl_base_1.0.safetensors",
            dest_subdir="checkpoints",
            size_gb=6.46,
        ),
        ModelSpec(
            repo="stabilityai/sdxl-vae",
            filename="sdxl_vae.safetensors",
            dest_subdir="vae",
            size_gb=0.31,
        ),
    ],
    # fp8 all-in-one: UNet + T5 + CLIP-L + VAE in one file, so it loads
    # via CheckpointLoaderSimple rather than a 3-loader split.
    "flux-dev": [
        ModelSpec(
            repo="Comfy-Org/flux1-dev",
            filename="flux1-dev-fp8.safetensors",
            dest_subdir="checkpoints",
            size_gb=16.06,
        ),
    ],
    "flux-schnell": [
        ModelSpec(
            repo="Comfy-Org/flux1-schnell",
            filename="flux1-schnell-fp8.safetensors",
            dest_subdir="checkpoints",
            size_gb=16.05,
        ),
    ],
}

CHECKPOINT_NAME: dict[str, str] = {
    "sdxl": "sd_xl_base_1.0.safetensors",
    "flux-dev": "flux1-dev-fp8.safetensors",
    "flux-schnell": "flux1-schnell-fp8.safetensors",
}


def resolve(profile: str, controlnet: bool = False,
            upscale: bool = True) -> list[ModelSpec]:
    """Full download list for a profile. Raises KeyError on unknown profile."""
    specs = list(PROFILES[profile])
    if upscale:
        specs.append(UPSCALER)
    if controlnet:
        specs.append(CONTROLNET_CANNY)
    return specs


def total_gb(specs: list[ModelSpec]) -> float:
    return round(sum(s.size_gb for s in specs), 2)

In [ ]:
%%writefile colab_studio/advice.py
"""Map detected GPU VRAM to a model profile, resolution ceiling and
ComfyUI launch flags.

Colab reassigns GPU tiers without warning, so nothing here may be
hardcoded to a subscription level. Thresholds are calibration targets
(spec E4) -- adjust the constants, not the structure.
"""
from __future__ import annotations

from dataclasses import dataclass, field

FLUX_DISK_GB = 16.06     # flux1-dev-fp8.safetensors, HEAD-verified
DISK_HEADROOM_GB = 8.0   # room for outputs, pip wheels, HF temp files

LOW_MAX_VRAM = 12.0
MID_MAX_VRAM = 20.0


@dataclass(frozen=True)
class Advice:
    tier: str
    profile: str
    max_side: int
    launch_flags: list[str] = field(default_factory=list)
    notes: list[str] = field(default_factory=list)


def recommend(vram_gb: float, disk_free_gb: float = 100.0) -> Advice:
    notes: list[str] = []

    if vram_gb < LOW_MAX_VRAM:
        tier, profile, max_side = "low", "sdxl", 768
        flags = ["--normalvram"]
        notes.append(
            f"{vram_gb:.1f} GB VRAM is tight for SDXL. Capped at 768px; "
            "expect OOM at 1024 with a large batch."
        )
    elif vram_gb < MID_MAX_VRAM:
        tier, profile, max_side = "mid", "sdxl", 1024
        flags = ["--normalvram"]
        notes.append(
            "SDXL at 1024px is comfortable. Flux fp8 (16 GB) will not fit "
            "alongside activations -- not offered at this tier."
        )
    else:
        tier, profile, max_side = "high", "flux-dev", 1024
        flags = ["--highvram"]
        notes.append("Enough VRAM for Flux dev fp8 and SDXL at 1024px.")

    # Disk guard: if model won't fit, downgrade profile but keep tier/flags.
    # Tier and launch_flags reflect available VRAM; profile reflects what will fit on disk.
    if profile.startswith("flux") and disk_free_gb < FLUX_DISK_GB + DISK_HEADROOM_GB:
        notes.append(
            f"Only {disk_free_gb:.0f} GB disk free; Flux needs "
            f"{FLUX_DISK_GB:.0f} GB plus headroom. Falling back to SDXL."
        )
        profile = "sdxl"

    return Advice(tier=tier, profile=profile, max_side=max_side,
                  launch_flags=flags, notes=notes)

In [ ]:
%%writefile colab_studio/workflows.py
"""API-format graph builders for ComfyUI.

API format is a flat dict: {node_id: {"class_type": str, "inputs": {...}}}.
Links are ["source_node_id", output_index] pairs.

Input key names come from INPUT_TYPES() on ComfyUI 0.10.0. They are not
guessable -- change them only against a fresh dump.
"""
from __future__ import annotations

Graph = dict


def _base(ckpt: str, prompt: str, negative: str, seed: int, steps: int,
          cfg: float, sampler: str, scheduler: str, denoise: float,
          prefix: str) -> Graph:
    """Shared spine: checkpoint -> two text encodes -> sampler -> decode -> save.

    Node "4" (the latent source) is deliberately left out; each builder
    supplies either EmptyLatentImage or VAEEncode.
    """
    return {
        "1": {"class_type": "CheckpointLoaderSimple",
              "inputs": {"ckpt_name": ckpt}},
        "2": {"class_type": "CLIPTextEncode",
              "inputs": {"text": prompt, "clip": ["1", 1]}},
        "3": {"class_type": "CLIPTextEncode",
              "inputs": {"text": negative, "clip": ["1", 1]}},
        "5": {"class_type": "KSampler",
              "inputs": {"model": ["1", 0], "seed": seed, "steps": steps,
                         "cfg": cfg, "sampler_name": sampler,
                         "scheduler": scheduler, "positive": ["2", 0],
                         "negative": ["3", 0], "latent_image": ["4", 0],
                         "denoise": denoise}},
        "6": {"class_type": "VAEDecode",
              "inputs": {"samples": ["5", 0], "vae": ["1", 2]}},
        "7": {"class_type": "SaveImage",
              "inputs": {"images": ["6", 0], "filename_prefix": prefix}},
    }


def _empty_latent(width: int, height: int, batch: int) -> Graph:
    return {"class_type": "EmptyLatentImage",
            "inputs": {"width": width, "height": height, "batch_size": batch}}


def sdxl_txt2img(ckpt: str, prompt: str, negative: str = "", seed: int = 0,
                 steps: int = 25, cfg: float = 7.0, width: int = 1024,
                 height: int = 1024, batch: int = 1,
                 sampler: str = "dpmpp_2m", scheduler: str = "karras") -> Graph:
    g = _base(ckpt, prompt, negative, seed, steps, cfg, sampler, scheduler,
              1.0, "colab/sdxl")
    g["4"] = _empty_latent(width, height, batch)
    return g


def flux_txt2img(ckpt: str, prompt: str, negative: str = "", seed: int = 0,
                 steps: int = 20, cfg: float = 1.0, width: int = 1024,
                 height: int = 1024, batch: int = 1,
                 guidance: float = 3.5) -> Graph:
    """Flux ignores CFG entirely -- it must be 1.0, with real guidance
    supplied by FluxGuidance. Any other cfg produces scorched output, so the
    parameter is overridden rather than trusted."""
    g = _base(ckpt, prompt, negative, seed, steps, 1.0, "euler", "simple",
              1.0, "colab/flux")
    g["4"] = _empty_latent(width, height, batch)
    g["8"] = {"class_type": "FluxGuidance",
              "inputs": {"conditioning": ["2", 0], "guidance": guidance}}
    g["5"]["inputs"]["positive"] = ["8", 0]
    return g


def img2img(ckpt: str, prompt: str, image: str, negative: str = "",
            seed: int = 0, steps: int = 25, cfg: float = 7.0,
            denoise: float = 0.6, sampler: str = "dpmpp_2m",
            scheduler: str = "karras") -> Graph:
    """`image` is a filename already present in the server's input/ dir --
    upload it first via ComfyClient.upload_image()."""
    g = _base(ckpt, prompt, negative, seed, steps, cfg, sampler, scheduler,
              denoise, "colab/img2img")
    g["10"] = {"class_type": "LoadImage", "inputs": {"image": image}}
    g["4"] = {"class_type": "VAEEncode",
              "inputs": {"pixels": ["10", 0], "vae": ["1", 2]}}
    return g


def upscale(ckpt: str, prompt: str, negative: str = "", seed: int = 0,
            steps: int = 25, cfg: float = 7.0, width: int = 1024,
            height: int = 1024, batch: int = 1,
            model_name: str = "4x-UltraSharp.pth",
            sampler: str = "dpmpp_2m", scheduler: str = "karras") -> Graph:
    """txt2img then a pure image-space upscale. No image input, so this is
    the one optional feature needing no upload path."""
    g = sdxl_txt2img(ckpt, prompt, negative, seed, steps, cfg, width, height,
                     batch, sampler, scheduler)
    g["11"] = {"class_type": "UpscaleModelLoader",
               "inputs": {"model_name": model_name}}
    g["12"] = {"class_type": "ImageUpscaleWithModel",
               "inputs": {"upscale_model": ["11", 0], "image": ["6", 0]}}
    g["7"]["inputs"]["images"] = ["12", 0]
    g["7"]["inputs"]["filename_prefix"] = "colab/upscale"
    return g


def controlnet_canny(ckpt: str, prompt: str, image: str, negative: str = "",
                     seed: int = 0, steps: int = 25, cfg: float = 7.0,
                     width: int = 1024, height: int = 1024, batch: int = 1,
                     strength: float = 0.8, low_threshold: float = 0.4,
                     high_threshold: float = 0.8,
                     control_net: str = "controlnet-canny-sdxl.safetensors",
                     sampler: str = "dpmpp_2m",
                     scheduler: str = "karras") -> Graph:
    """SDXL only. Canny is a core node -- no comfyui_controlnet_aux needed.

    ControlNetApplyAdvanced emits BOTH conditionings, so the sampler's
    positive and negative must be rewired to outputs 0 and 1 of the same
    node. Rewiring only positive is a silent correctness bug.
    """
    g = sdxl_txt2img(ckpt, prompt, negative, seed, steps, cfg, width, height,
                     batch, sampler, scheduler)
    g["10"] = {"class_type": "LoadImage", "inputs": {"image": image}}
    g["13"] = {"class_type": "Canny",
               "inputs": {"image": ["10", 0], "low_threshold": low_threshold,
                          "high_threshold": high_threshold}}
    g["14"] = {"class_type": "ControlNetLoader",
               "inputs": {"control_net_name": control_net}}
    g["15"] = {"class_type": "ControlNetApplyAdvanced",
               "inputs": {"positive": ["2", 0], "negative": ["3", 0],
                          "control_net": ["14", 0], "image": ["13", 0],
                          "strength": strength, "start_percent": 0.0,
                          "end_percent": 1.0}}
    g["5"]["inputs"]["positive"] = ["15", 0]
    g["5"]["inputs"]["negative"] = ["15", 1]
    g["7"]["inputs"]["filename_prefix"] = "colab/controlnet"
    return g

In [ ]:
%%writefile colab_studio/fetch.py
"""Download ModelSpecs into a ComfyUI models/ tree.

Uses local_dir= so huggingface_hub writes straight to the destination.
The previous implementation downloaded to the HF cache and then copied,
doubling disk use -- fatal for a 16 GB checkpoint on a Colab VM.
"""
from __future__ import annotations

import os
import shutil  # noqa: F401 -- referenced only via monkeypatch in fetch_test.py
from typing import Callable

from huggingface_hub import hf_hub_download

from colab_studio.registry import ModelSpec

Emit = Callable[[str], None]


def _noop(_: str) -> None:
    return None


def download(spec: ModelSpec, models_dir: str, emit: Emit | None = None) -> str:
    """Fetch one spec. Returns the final on-disk path. Idempotent."""
    log = emit or _noop
    dest_dir = os.path.join(models_dir, spec.dest_subdir)
    os.makedirs(dest_dir, exist_ok=True)
    final = os.path.join(dest_dir, spec.target_name)

    if os.path.exists(final):
        log(f"[=] {spec.target_name} already present, skipping")
        return final

    log(f"[+] {spec.target_name} ({spec.size_gb:.2f} GB) from {spec.repo}")
    got = hf_hub_download(
        repo_id=spec.repo,
        filename=spec.filename,
        local_dir=dest_dir,
    )

    # Nested filenames land in a subtree, and diffusers-layout repos all use
    # the same generic name; flatten and rename to the target.
    if os.path.abspath(got) != os.path.abspath(final):
        os.replace(got, final)
        stray = os.path.dirname(got)
        while os.path.abspath(stray) != os.path.abspath(dest_dir):
            try:
                os.rmdir(stray)
            except OSError:
                break
            stray = os.path.dirname(stray)

    log(f"[v] {spec.target_name} ready")
    return final


def download_all(specs: list[ModelSpec], models_dir: str,
                 emit: Emit | None = None) -> list[str]:
    return [download(s, models_dir, emit) for s in specs]

In [ ]:
%%writefile colab_studio/client.py
"""HTTP client for a running ComfyUI server.

Deliberately talks HTTP rather than importing comfy: comfy/cli_args.py:236
parses sys.argv when args_parsing is enabled, and comfy/model_management.py:238
probes the GPU at import time. Both are hostile inside a notebook kernel.
"""
from __future__ import annotations

import os
import time
import uuid

import requests


class ComfyError(RuntimeError):
    """Server rejected a request. Carries node_errors when present."""


class ComfyClient:
    def __init__(self, base_url: str = "http://127.0.0.1:8188") -> None:
        self.base_url = base_url.rstrip("/")
        self.client_id = str(uuid.uuid4())

    def wait_ready(self, timeout: float = 180.0, interval: float = 1.0) -> bool:
        """Poll /system_stats until the server answers. Start the tunnel only
        after this returns True, or the public URL 502s."""
        deadline = time.time() + timeout
        while time.time() < deadline:
            try:
                r = requests.get(f"{self.base_url}/system_stats", timeout=5)
                if r.status_code == 200:
                    return True
            except (requests.exceptions.MissingSchema,
                    requests.exceptions.InvalidSchema,
                    requests.exceptions.InvalidURL) as err:
                raise ComfyError(f"invalid base_url {self.base_url!r}: {err}") from err
            except requests.RequestException:
                pass
            time.sleep(interval)
        return False

    def upload_image(self, path: str) -> str:
        """Upload to the server's input/ dir. Returns the name to put in
        LoadImage.inputs.image."""
        with open(path, "rb") as fh:
            r = requests.post(
                f"{self.base_url}/upload/image",
                files={"image": (os.path.basename(path), fh)},
                data={"overwrite": "true"},
                timeout=120,
            )
        if r.status_code != 200:
            raise ComfyError(f"upload failed ({r.status_code}): {r.text[:300]}")
        return r.json()["name"]

    def submit(self, graph: dict) -> str:
        r = requests.post(
            f"{self.base_url}/prompt",
            json={"prompt": graph, "client_id": self.client_id},
            timeout=60,
        )
        if r.status_code != 200:
            try:
                payload = r.json()
            except ValueError:
                raise ComfyError(f"submit failed ({r.status_code}): {r.text[:300]}")
            raise ComfyError(
                f"submit rejected: {payload.get('error')} "
                f"node_errors={payload.get('node_errors')}"
            )
        return r.json()["prompt_id"]

    def wait_result(self, prompt_id: str, timeout: float = 600.0,
                    interval: float = 1.0) -> list[dict]:
        """Poll /history until outputs appear. Returns image refs."""
        deadline = time.time() + timeout
        while time.time() < deadline:
            r = requests.get(f"{self.base_url}/history/{prompt_id}", timeout=15)
            if r.status_code == 200:
                hist = r.json().get(prompt_id)
                if hist and hist.get("outputs"):
                    refs: list[dict] = []
                    for node_out in hist["outputs"].values():
                        refs.extend(node_out.get("images", []))
                    if refs:
                        return refs
            time.sleep(interval)
        raise TimeoutError(f"no outputs for {prompt_id} within {timeout}s")

    def fetch_image(self, ref: dict) -> bytes:
        r = requests.get(
            f"{self.base_url}/view",
            params={"filename": ref["filename"],
                    "subfolder": ref.get("subfolder", ""),
                    "type": ref.get("type", "output")},
            timeout=120,
        )
        if r.status_code != 200:
            raise ComfyError(f"view failed ({r.status_code})")
        return r.content

    def generate(self, graph: dict, timeout: float = 600.0) -> list[bytes]:
        """submit -> wait -> fetch. The whole inline-cell path in one call."""
        pid = self.submit(graph)
        return [self.fetch_image(ref) for ref in self.wait_result(pid, timeout)]

In [ ]:
%%writefile colab_studio/launch.py
"""Background the ComfyUI server so the notebook kernel stays usable.

The original notebook ended with `!python3 main.py`, which never returns --
every cell after it was unreachable. Backgrounding is what makes the
generate/log/ops cells exist at all.

Ordering matters: server -> wait_ready() -> tunnel. Starting the tunnel
first prints a URL that 502s until the server finishes booting.
"""
from __future__ import annotations

import os
import re
import subprocess
import sys
import time

CLOUDFLARED = "/usr/local/bin/cloudflared"
TUNNEL_RE = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")


def start_server(comfy_dir: str, flags: list[str], log_path: str,
                 port: int = 8188, python: str | None = None) -> subprocess.Popen:
    """Launch main.py detached, stdout+stderr to log_path. Returns at once."""
    exe = python or sys.executable
    cmd = [exe, "main.py", "--listen", "127.0.0.1", "--port", str(port), *flags]
    log = open(log_path, "wb")
    return subprocess.Popen(
        cmd, cwd=comfy_dir, stdout=log, stderr=subprocess.STDOUT,
        start_new_session=True,
    )


def start_tunnel(port: int, log_path: str, timeout: float = 40.0,
                 interval: float = 1.0) -> str | None:
    """Start cloudflared and scrape the public URL out of its log.

    Call only after ComfyClient.wait_ready() returns True.
    """
    if os.path.exists(log_path):
        os.remove(log_path)
    log = open(log_path, "wb")
    try:
        proc = subprocess.Popen(
            [CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{port}"],
            stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
        )
    except FileNotFoundError:
        return None
    deadline = time.time() + timeout
    while time.time() < deadline:
        if proc.poll() not in (None,):
            return None
        try:
            with open(log_path, "r", errors="ignore") as fh:
                m = TUNNEL_RE.search(fh.read())
            if m:
                return m.group(0)
        except FileNotFoundError:
            pass
        time.sleep(interval)
    proc.terminate()
    return None


def tail(log_path: str, n: int = 40) -> str:
    """Last n lines of a logfile. Empty string if it does not exist yet."""
    try:
        with open(log_path, "r", errors="ignore") as fh:
            return "\n".join(fh.read().splitlines()[-n:])
    except FileNotFoundError:
        return ""

In [ ]:
#@title 5. Choose profile and download models
import sys
sys.path.insert(0, COMFY_DIR)
from colab_studio.advice import recommend
from colab_studio.registry import resolve, total_gb, CHECKPOINT_NAME
from colab_studio.fetch import download_all

ADVICE = recommend(VRAM_GB, DISK_FREE_GB)
PROFILE = ADVICE.profile if IMAGE_MODEL == "auto" else IMAGE_MODEL
LAUNCH_FLAGS = ADVICE.launch_flags
MAX_SIDE = ADVICE.max_side

print(f"tier={ADVICE.tier}  profile={PROFILE}  max_side={MAX_SIDE}")
print(f"launch flags: {' '.join(LAUNCH_FLAGS) or '(none)'}")
for n in ADVICE.notes:
    print(" -", n)

SPECS = resolve(PROFILE, controlnet=CONTROLNET, upscale=UPSCALER)
print(f"\nDownloading {len(SPECS)} files, {total_gb(SPECS)} GB total")
download_all(SPECS, os.path.join(COMFY_DIR, "models"), emit=print)
CKPT = CHECKPOINT_NAME[PROFILE]

In [ ]:
#@title 6. Write workflows into the ComfyUI sidebar
import json, os
from colab_studio import workflows

WF_DIR = os.path.join(COMFY_DIR, "user", "default", "workflows")
os.makedirs(WF_DIR, exist_ok=True)
API_DIR = "/content/wf_api"
os.makedirs(API_DIR, exist_ok=True)

built = {
    "sdxl_txt2img": workflows.sdxl_txt2img(CKPT, "a prompt"),
    "upscale": workflows.upscale(CKPT, "a prompt"),
}
if PROFILE.startswith("flux"):
    built["flux_txt2img"] = workflows.flux_txt2img(CKPT, "a prompt")
if CONTROLNET:
    built["controlnet_canny"] = workflows.controlnet_canny(CKPT, "a prompt", image="input.png")

for name, graph in built.items():
    with open(os.path.join(API_DIR, f"{name}.json"), "w") as fh:
        json.dump(graph, fh, indent=1)
print("workflows written:", ", ".join(built))

In [ ]:
#@title 7. Launch server (backgrounded) then open the tunnel
import os
from colab_studio.client import ComfyClient
from colab_studio.launch import start_server, start_tunnel

if not os.path.isfile("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

SERVER_LOG = "/content/comfyui.log"
TUNNEL_LOG = "/content/cloudflared.log"

SERVER = start_server(COMFY_DIR, LAUNCH_FLAGS, SERVER_LOG, port=PORT)
CLIENT = ComfyClient(f"http://127.0.0.1:{PORT}")

print("waiting for server...")
if CLIENT.wait_ready(timeout=300):
    # Tunnel only AFTER readiness, or the URL 502s.
    URL = start_tunnel(PORT, TUNNEL_LOG)
    print("ComfyUI ready.  Public URL:", URL or "(tunnel failed, see cell 10)")
else:
    print("server did not come up - run the log cell below")

In [ ]:
#@title 8. Generate an image here (no tunnel needed)
from IPython.display import Image, display
from colab_studio import workflows

prompt = "a lighthouse in a storm, dramatic light"  #@param {type:"string"}
negative = "blurry, watermark, text"  #@param {type:"string"}
steps = 25  #@param {type:"slider", min:4, max:60, step:1}
cfg = 7.0  #@param {type:"number"}
seed = 0  #@param {type:"integer"}
width = 1024  #@param {type:"integer"}
height = 1024  #@param {type:"integer"}
use_upscaler = False  #@param {type:"boolean"}

width, height = min(width, MAX_SIDE), min(height, MAX_SIDE)
kw = dict(prompt=prompt, negative=negative, seed=seed, steps=steps,
          width=width, height=height)
if use_upscaler:
    graph = workflows.upscale(CKPT, cfg=cfg, **kw)
elif PROFILE.startswith("flux"):
    graph = workflows.flux_txt2img(CKPT, **kw)
else:
    graph = workflows.sdxl_txt2img(CKPT, cfg=cfg, **kw)

for i, data in enumerate(CLIENT.generate(graph)):
    path = f"/content/gen_{i}.png"
    with open(path, "wb") as fh:
        fh.write(data)
    display(Image(filename=path))

In [ ]:
#@title 8b. Image-to-image / ControlNet (upload or URL)
import os, requests
from IPython.display import Image, display
from colab_studio import workflows

source = "upload"  #@param ["upload", "url"]
image_url = ""  #@param {type:"string"}
mode = "img2img"  #@param ["img2img", "controlnet"]
prompt2 = "an oil painting of the same scene"  #@param {type:"string"}
denoise = 0.6  #@param {type:"slider", min:0.1, max:1.0, step:0.05}

local = "/content/source_image.png"
if source == "upload":
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    with open(local, "wb") as fh:
        fh.write(up[name])
else:
    with open(local, "wb") as fh:
        fh.write(requests.get(image_url, timeout=60).content)

server_name = CLIENT.upload_image(local)
if mode == "controlnet":
    graph = workflows.controlnet_canny(CKPT, prompt2, image=server_name)
else:
    graph = workflows.img2img(CKPT, prompt2, image=server_name, denoise=denoise)

for i, data in enumerate(CLIENT.generate(graph)):
    path = f"/content/edit_{i}.png"
    with open(path, "wb") as fh:
        fh.write(data)
    display(Image(filename=path))

In [ ]:
#@title 9. Server log
from colab_studio.launch import tail
lines = 60  #@param {type:"integer"}
print(tail(SERVER_LOG, n=lines) or "(log empty)")

In [ ]:
#@title 10. Ops - restart, free VRAM, disk, re-tunnel
import os, shutil, requests
action = "disk usage"  #@param ["disk usage", "free VRAM", "restart server", "re-tunnel", "list models"]

if action == "disk usage":
    u = shutil.disk_usage("/content")
    print(f"{u.free/2**30:.1f} GB free of {u.total/2**30:.1f} GB")
elif action == "free VRAM":
    requests.post(f"http://127.0.0.1:{PORT}/free",
                  json={"unload_models": True, "free_memory": True}, timeout=30)
    print("asked ComfyUI to unload models")
elif action == "restart server":
    from colab_studio.launch import start_server
    SERVER.terminate(); SERVER.wait(timeout=30)
    SERVER = start_server(COMFY_DIR, LAUNCH_FLAGS, SERVER_LOG, port=PORT)
    print("restarted:", CLIENT.wait_ready(timeout=300))
elif action == "re-tunnel":
    from colab_studio.launch import start_tunnel
    print("URL:", start_tunnel(PORT, TUNNEL_LOG))
else:
    for root, _, fs in os.walk(os.path.join(COMFY_DIR, "models")):
        for f in fs:
            p = os.path.join(root, f)
            print(f"{os.path.getsize(p)/2**30:6.2f} GB  {os.path.relpath(p, COMFY_DIR)}")

## Handbook

### Model sizes and what fits

| Profile | Download | Needs | Notes |
|---|---|---|---|
| `sdxl` | 6.8 GB | ~12 GB VRAM at 1024px | Best all-rounder on a T4 |
| `flux-dev` | 16.1 GB | ~20 GB VRAM | All-in-one fp8: UNet + T5 + CLIP-L + VAE |
| `flux-schnell` | 16.1 GB | ~20 GB VRAM | 4-step; much faster, slightly lower fidelity |
| upscaler | 0.06 GB | negligible | 4x-UltraSharp, image-space |
| ControlNet canny | 2.33 GB | +2 GB VRAM | SDXL only |

### Settings that matter

| Model | steps | cfg | sampler / scheduler |
|---|---|---|---|
| SDXL | 25-30 | 6-8 | `dpmpp_2m` / `karras` |
| Flux dev | 20-25 | **1.0** | `euler` / `simple`, guidance 3.5 |
| Flux schnell | **4** | **1.0** | `euler` / `simple` |

**Flux cfg must be 1.0.** Flux does not use classifier-free guidance; real
guidance rides on the `FluxGuidance` node. Any other cfg scorches the image.

### When it breaks

| Symptom | Cause | Fix |
|---|---|---|
| Tunnel URL 502s | Server still booting | Rerun cell 7; it waits for readiness first |
| `CUDA out of memory` | Resolution or batch too high | Drop to 768px, batch 1, run "free VRAM" in cell 10 |
| `No such file or directory: ...safetensors` | Download interrupted | Rerun cell 5 - it skips completed files |
| Disk full mid-download | Flux is 16 GB | Use `sdxl`, or set `PERSIST="off"` to reclaim Drive space |
| Generate cell hangs | Server died | Check cell 9 log, then "restart server" in cell 10 |
| Session dropped | Colab idle timeout | Rerun all; with `PERSIST` on, models and outputs survive |

### Adding a LoRA without leaving Colab

```python
from huggingface_hub import hf_hub_download
hf_hub_download(repo_id="OWNER/REPO", filename="lora.safetensors",
                local_dir=f"{COMFY_DIR}/models/loras")
```

Then use the `LoraLoader` node in the tunnel UI (it is a core node, already
available).